# Task 3 · HypER's distilled checker (Phi-3 + LoRA) as a second opinion on Stage 2

**Why.** HypER (Vasu et al., EMNLP 2025) has two halves: a large model builds and scores citation chains (the
*teacher*), and a small model, **Phi-3-mini-128k-instruct**, is fine-tuned on those chains to check them (the
*student*). Our pipeline ran only the teacher half (Claude scored the 0/1/2 relevance of every paper citing MVFA). This
notebook trains the student on HypER's released chains and uses it to re-check our Stage 2:

1. **Relevance:** does the student give the same 0/1/2 score as our judge to the 158 papers citing MVFA (and the 9
   second-hop papers)? Reported as exact agreement, weighted kappa and a confusion matrix.
2. **Chains:** does it find each of our six chains valid?
3. **Seeds:** does it find that each chain supports the hypothesis of the seed idea built on it?

It also scores the base Phi-3 and the fine-tuned one on HypER's own test split, to check the training worked (HypER
reports average F1 0.468 → 0.616 and 1-hop F1 0.17 → 0.77).

**Runtime:** A100 GPU. Training 1 epoch on all 9,102 examples takes roughly 2–3 hours; `LIMIT = 2000` gives a quick
version in about 40 minutes. Checkpoints go to Google Drive every 100 steps, so a disconnected session continues
where it stopped when you re-run the training cell.

**Honest caveat:** HypER's chains are clinical-trial literature (PubMed), not computer vision. Agreement on our MVFA
papers therefore tests how well the checker transfers to a new domain, and disagreement is not proof that our judge
was wrong.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (for checkpoints) and code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/task3_hyper_phi3"      # adapter, checkpoints and evaluation files

In [ ]:
import os, subprocess
REPO = "https://github.com/Divyam10113/Intro-to-LLMs.git"   # for a private repo: https://<token>@github.com/Divyam10113/Intro-to-LLMs.git
BRANCH = "task3_kevalina"
ZIP = "/content/task3_run3.zip"                              # alternative: upload the zip to the file panel

if not os.path.exists("/content/repo/task3"):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/content/repo"], capture_output=True, text=True)
    if r.returncode != 0:
        print("clone failed, trying the zip:", r.stderr[-300:])
        os.makedirs("/content/repo", exist_ok=True)
        subprocess.run(["unzip", "-q", ZIP, "-d", "/content/repo"], check=True)
if not os.path.exists("/content/HypER"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rosnikv/HypER.git", "/content/HypER"], check=True)
%cd /content/repo/task3
HYPER_DATA = "/content/HypER/dataset/training_data/balanced_splits_w_hyp"

In [ ]:
!pip install -q -U transformers peft accelerate
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))"

## 2. Build the training data

Rebuilds HypER's multi-task data from its released chains (same instructions and labelling rules as
`multi-task-data_prep.py`) and the same task formats for our MVFA data. Prints the number of examples per task.

In [ ]:
!python colab/hyper_checker.py $HYPER_DATA

## 3. Fine-tune Phi-3 with LoRA

HypER's released settings: LoRA r=8, alpha=16, dropout 0.05 on all linear layers, lr 2e-4, cosine schedule, loss on
the answer only. One GPU forces 1 epoch (HypER: 10 on 8 GPUs) and a maximum length of 8,192 tokens.

Set `LIMIT = 2000` for a quick run, `LIMIT = 0` for all examples. Re-run this cell after a disconnect to resume.

In [ ]:
EPOCHS = 1
LIMIT = 0
MAX_LEN = 8192          # use 6144 on an L4 GPU
!python colab/train_hyper_phi3.py train --hyper-data $HYPER_DATA --out $OUT --epochs $EPOCHS --limit $LIMIT --max-len $MAX_LEN

## 4. Check the training on HypER's test split

A stratified subset of 300 test examples (100 per task), for the base model and the fine-tuned one. About 10 minutes
each. Compare with HypER's Table: average F1 0.468 (base) → 0.616 (fine-tuned).

In [ ]:
EVAL_N = 300
!python colab/train_hyper_phi3.py eval --hyper-data $HYPER_DATA --out $OUT --limit $EVAL_N --base | tail -n 40
!python colab/train_hyper_phi3.py eval --hyper-data $HYPER_DATA --out $OUT --limit $EVAL_N | tail -n 40

## 5. Check our Stage 2 with the trained checker

In [ ]:
!python colab/train_hyper_phi3.py mvfa --task3 . --out $OUT
!cp $OUT/eval_lora.json outputs/hypercheck_eval_lora.json
!cp $OUT/eval_base.json outputs/hypercheck_eval_base.json
!python stage2b_hyper_check.py

In [ ]:
import json
S = json.load(open("outputs/stage2b_hyper_check.json"))
for k in ("hop1", "hop2"):
    r = S["relevance"][k]
    print(f"{k}: same score as our judge for {r['exact']:.0%} of {r['parsed']} papers, weighted kappa {r['qwk']}")
    print("   ours  ", r["ours_distribution"], "\n   checker", r["checker_distribution"])
print("chain starters the checker scores 0:", S["relevance"]["chain_starters_scored_0_by_checker"] or "none")
print("chains:", [(c["chain"], c["validity"]) for c in S["chains"]])
print("seeds supported by their chain:", sum(s["supported"] for s in S["seeds"]), "of", len(S["seeds"]))
for name in ("hyper_test_base", "hyper_test_lora"):
    if name in S:
        m = S[name]
        avg = sum(v["macro_f1"] for v in m.values()) / len(m)
        print(name, {t: v["macro_f1"] for t, v in m.items()}, f"average {avg:.3f}")

## 6. Save the results

Download the zip and unzip it **inside your local `task3/` folder** (it adds `outputs/hypercheck_phi3.json`, the two
evaluation files and `outputs/stage2b_hyper_check.json`). Then `python3 make_reports.py` adds a section on the checker
to the documentation. The LoRA adapter stays in your Drive (`task3_hyper_phi3/adapter`, about 50 MB).

In [ ]:
import zipfile
z = "/content/hypercheck_results.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["outputs/hypercheck_phi3.json", "outputs/hypercheck_eval_lora.json", "outputs/hypercheck_eval_base.json",
              "outputs/stage2b_hyper_check.json"]:
        if os.path.exists(f):
            zf.write(f)
    if os.path.exists(f"{OUT}/train_info.json"):
        zf.write(f"{OUT}/train_info.json", "outputs/hypercheck_train_info.json")
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print("download from the file panel:", e)